In [5]:
!pip install -q transformers datasets sentence-transformers

In [6]:
import torch
import numpy as np

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    pipeline
)

from sentence_transformers import SentenceTransformer, util

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)

train = dataset["train"]

print(train)

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
    num_rows: 2000
})


In [8]:
train = train.map(
    lambda x: {
        "combined_text": x["prompt"] + " " + x["A"]
    }
)

print(train[51]["combined_text"])

print(len(train[51]["combined_text"]))

Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what is their color and composition? among the listed options. Class L dwarfs are hotter than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are bright blue in color and are brightest in ultraviolet. Their atmosphere is hot enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to support hydrogen fusion and are therefore stars, but most are of substellar mass and are therefore brown dwarfs.
614


In [9]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(tokenizer.vocab_size)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

30522


In [10]:
print(tokenizer.sep_token_id)

102


In [13]:
print(type(train))
print(type(train["prompt"]))
print(train["prompt"][:3])

<class 'datasets.arrow_dataset.Dataset'>
<class 'datasets.arrow_dataset.Column'>
["Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.", 'What is accelerator-based light-ion fusion?', 'Determine the correct option: What is the term used in astrophysics to describe light-matter interactions resulting in energy shifts in the radiation field? among the listed options.']


In [14]:
prompts = train[:]["prompt"]        # Converts the column to a Python list

encodings = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print(encodings["input_ids"].shape)

torch.Size([2000, 128])


In [15]:
hidden_size = 768
num_heads = 12

print(hidden_size // num_heads)

64


In [16]:
model = AutoModel.from_pretrained("bert-base-uncased")
model.eval()

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [17]:
text = train[0]["prompt"]

inputs = tokenizer(
    text,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = model(**inputs)

print(outputs.last_hidden_state.shape)

torch.Size([1, 31, 768])


In [18]:
cls_vector = outputs.last_hidden_state[0,0]

print(round(cls_vector[:5].sum().item(),4))

-1.2001


In [19]:
attention_model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

sentence = "Light-ion fusion is a technique."

inputs = tokenizer(
    sentence,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = attention_model(**inputs)

tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0]
)

print(tokens)

fusion_index = tokens.index("fusion")

attention = outputs.attentions[-1]

value = attention[0,0,0,fusion_index]

print(round(value.item(),4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
0.1025


In [20]:
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (3 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [21]:
prompt = train[0]["prompt"]
optionB = train[0]["B"]

embeddings = embedder.encode(
    [prompt, optionB],
    convert_to_tensor=True
)

score = util.cos_sim(
    embeddings[0],
    embeddings[1]
)

print(round(score.item(),4))

0.7658


In [23]:
def map3(actual, predicted):

    score = 0

    for a, preds in zip(actual, predicted):

        if a in preds:

            rank = preds.index(a) + 1

            score += 1/rank

    return score/len(actual)

In [24]:
labels = ["A","B","C","D","E"]

predictions_tfidf = []

for row in train:

    docs = [
        row["prompt"],
        row["A"],
        row["B"],
        row["C"],
        row["D"],
        row["E"]
    ]

    tfidf = TfidfVectorizer()

    matrix = tfidf.fit_transform(docs)

    sims = cosine_similarity(
        matrix[0],
        matrix[1:]
    )[0]

    ranked = np.argsort(sims)[::-1]

    predictions_tfidf.append(
        [labels[i] for i in ranked[:3]]
    )

actual = train["answer"]

print(map3(actual,predictions_tfidf))

0.3269166666666673


In [25]:
predictions_minilm = []

for row in train:

    prompt_embedding = embedder.encode(
        row["prompt"],
        convert_to_tensor=True
    )

    option_embeddings = embedder.encode(
        [
            row["A"],
            row["B"],
            row["C"],
            row["D"],
            row["E"]
        ],
        convert_to_tensor=True
    )

    similarities = util.cos_sim(
        prompt_embedding,
        option_embeddings
    )[0]

    ranked = torch.argsort(
        similarities,
        descending=True
    )

    predictions_minilm.append(
        [labels[i] for i in ranked[:3]]
    )

print("MiniLM MAP@3")

print(map3(actual,predictions_minilm))

MiniLM MAP@3
0.42308333333333487


In [26]:
count = 0

for ans, tfidf, mini in zip(
    actual,
    predictions_tfidf,
    predictions_minilm
):

    if ans not in tfidf and ans in mini:
        count += 1

print(count)

564


In [27]:
classifier = pipeline(
    "zero-shot-classification"
)

result = classifier(
    train[1]["prompt"],
    candidate_labels=[
        train[1]["A"],
        train[1]["B"],
        train[1]["C"]
    ]
)

print(result)

print(round(result["scores"][0],4))

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion

In [28]:
result_softmax = classifier(
    train[1]["prompt"],
    candidate_labels=[
        train[1]["A"],
        train[1]["B"],
        train[1]["C"]
    ]
)

sum_softmax = sum(result_softmax["scores"])

result_sigmoid = classifier(
    train[1]["prompt"],
    candidate_labels=[
        train[1]["A"],
        train[1]["B"],
        train[1]["C"]
    ],
    multi_label=True
)

sum_sigmoid = sum(result_sigmoid["scores"])

print(abs(sum_softmax - sum_sigmoid))

0.9994903629012697


In [30]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

prompt = (
    f"Question: {train[0]['prompt']}. "
    f"Is the correct answer A: {train[0]['A']} "
    f"or B: {train[0]['B']}? "
    f"Answer with just the letter A or B."
)

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=5
)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(answer)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

B
